In [ ]:
#| export: sales, summary

import pandas as pd
import numpy as np

# Simulate sales data (in production, this would be a DB query)
np.random.seed(42)
regions = ['Northeast', 'Southeast', 'Midwest', 'West', 'International']
quarters = ['Q1', 'Q2', 'Q3', 'Q4']

rows = []
for region in regions:
    base = np.random.randint(800, 2000)
    for i, quarter in enumerate(quarters):
        growth = 1 + np.random.normal(0.05, 0.03) * (i + 1)
        revenue = int(base * growth * 1000)
        rows.append({
            'region': region,
            'quarter': quarter,
            'revenue': revenue,
            'deals': np.random.randint(20, 100),
            'avg_deal_size': revenue // np.random.randint(20, 100)
        })

sales = pd.DataFrame(rows)

summary = {
    'total_revenue': int(sales['revenue'].sum()),
    'avg_deal_size': int(sales['avg_deal_size'].mean()),
    'total_deals': int(sales['deals'].sum()),
    'top_region': sales.groupby('region')['revenue'].sum().idxmax(),
    'growth_pct': round(
        (sales[sales.quarter == 'Q4'].revenue.sum() /
         sales[sales.quarter == 'Q1'].revenue.sum() - 1) * 100, 1
    )
}

In [ ]:
#| export: forecast

# Simple linear forecast for next quarter
from numpy.polynomial import polynomial as P

quarterly_totals = sales.groupby('quarter')['revenue'].sum().values
x = np.arange(len(quarterly_totals))
coeffs = P.polyfit(x, quarterly_totals, deg=1)
next_q = P.polyval(len(quarterly_totals), coeffs)

forecast = {
    'next_quarter': 'Q5 (projected)',
    'projected_revenue': int(next_q),
    'confidence': 'moderate',
    'trend': 'upward' if coeffs[1] > 0 else 'downward',
    'quarterly_totals': [
        {'quarter': f'Q{i+1}', 'revenue': int(v)}
        for i, v in enumerate(quarterly_totals)
    ] + [{'quarter': 'Q5 (proj)', 'revenue': int(next_q)}]
}